# Import data

In [5]:
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [6]:
import kagglehub

path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")

print("Path to dataset files:", path)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\arashnic\book-recommendation-dataset\versions\3


In [7]:
df = pd.read_csv("result.csv", on_bad_lines='skip')
df_book = pd.read_csv(f"{path}/books.csv", on_bad_lines='skip')

df_book = df_book[df_book['Book-Title'].isin(df['title'])]
df_book = df_book.drop(['Publisher','Image-URL-S', 'Image-URL-M', 'Image-URL-L'], axis=1)
ratings = pd.read_csv(f"{path}/Ratings.csv", on_bad_lines='skip')


C:\Users\User\AppData\Local\Temp\ipykernel_27796\3845761129.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_book = pd.read_csv(f"{path}/books.csv", on_bad_lines='skip')


# Handling missing data

In [8]:
# Fill missing values
df_book['Book-Author'] = df_book['Book-Author'].fillna('Unknown Author')


# Clean year data
df_book['Year-Of-Publication'] = pd.to_numeric(df_book['Year-Of-Publication'], errors='coerce')
df_book = df_book.dropna(subset=['Year-Of-Publication'])

df_book.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
dtype: int64

# Colloborative Filtering

In [9]:
# Colloborative Filtering -Based Recommendation System

merged_df = pd.merge(df_book,ratings,on='ISBN')

combine_book_rating = merged_df.dropna(axis = 0, subset = ['Book-Title'])
book_ratingCount = (combine_book_rating.
     groupby(by = ['Book-Title'])['Book-Rating'].
     count().
     reset_index().
     rename(columns = {'Book-Rating': 'totalRatingCount'})
     [['Book-Title', 'totalRatingCount']]
    )

rating_with_totalRatingCount = combine_book_rating.merge(book_ratingCount, left_on = 'Book-Title', right_on = 'Book-Title', how = 'left')

popularity_threshold = 20
rating_popular_book= rating_with_totalRatingCount.query('totalRatingCount >= @popularity_threshold')
book_features_df=rating_popular_book.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating').fillna(0)
book_features_df_matrix = csr_matrix(book_features_df.values)
model_knn = NearestNeighbors(metric = 'cosine', algorithm = 'brute')
model_knn.fit(book_features_df_matrix)

query_index = np.random.choice(book_features_df.shape[0])
distances, indices = model_knn.kneighbors(book_features_df.iloc[query_index,:].values.reshape(1, -1), n_neighbors = 6)

for i in range(0, len(distances.flatten())):
    if i == 0:
        print('Query index:', query_index)
        print('Recommendations for ', book_features_df.index[query_index])
    else:
        print('{0}: {1}, with distance of {2}:'.format(i, book_features_df.index[indices.flatten()[i]], distances.flatten()[i]))

Query index: 137
Recommendations for  The Trial
1: Red, with distance of 0.8875134558374406:
2: American Pastoral, with distance of 0.924438354555722:
3: On the Road, with distance of 0.9324552835636172:
4: The Odyssey, with distance of 0.9395706858027231:
5: The Big Sleep, with distance of 0.9449961085653124:


In [19]:
top_books = book_ratingCount.sort_values('totalRatingCount', ascending=False).head(5)

# Save a model

In [21]:
joblib.dump(model_knn, 'model_knn.pkl')
book_features_df.to_pickle('book_features_df.pkl')
top_books.to_pickle("top_books.pkl")

In [ ]:
book_features_df = pd.read_pickle('book_features_df.pkl')
top_books = pd.read_pickle("top_books.pkl")

In [22]:
print(len(book_features_df), len(top_books))

152 5
